# Wireless Networking and Mobile Computing

### Prerequisites
Run the code cell below once before starting. You can **collapse it afterwards**; the exercise cells below use the functions defined here.

In [1]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib numpy ipywidgets --quiet

# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import IntSlider, FloatSlider, HBox, VBox, interactive_output, Layout
from IPython.display import display


# --- Visualization helpers (no need to modify) ---

def _style_axis(ax):
    ax.grid()
    ax.set_facecolor((.2, 0.2, 0.3))


def _add_legend(ax, loc="upper right"):
    ax.legend(
        fancybox=True,
        framealpha=1,
        loc=loc,
        facecolor=(.6, 0.61, 0.8)
    )


def show_two_frequency_fft_widget(initial_f1, initial_f2, signal_generator):
    """Interactive visualization of two tones, their sum, and its FFT."""

    slider_max = max(150, int(initial_f1), int(initial_f2))

    f1_slider = IntSlider(
        value=int(initial_f1),
        min=1,
        max=slider_max,
        step=1,
        description="f1 [Hz]",
        continuous_update=True,
        style={"description_width": "70px"},
        layout=Layout(width="320px")
    )

    f2_slider = IntSlider(
        value=int(initial_f2),
        min=1,
        max=slider_max,
        step=1,
        description="f2 [Hz]",
        continuous_update=True,
        style={"description_width": "70px"},
        layout=Layout(width="320px")
    )

    def _plot(f1_value, f2_value):
        (
            t,
            signal_f1,
            signal_f2,
            combined_signal,
            freq,
            fft_magnitude,
            N_current
        ) = signal_generator(f1_value, f2_value)

        fig, axs = plt.subplots(2, 2, figsize=(10, 7))
        theLineWidth = 1

        # Individual component at f1
        axs[0, 0].plot(
            t, signal_f1,
            label=f"f1 = {f1_value} Hz",
            color="#4C78A8",
            linewidth=theLineWidth
        )
        axs[0, 0].set_xlabel("Time [s]")
        axs[0, 0].set_ylabel("Amplitude")
        axs[0, 0].set_title("First Frequency Component")
        _style_axis(axs[0, 0])
        _add_legend(axs[0, 0])

        # Individual component at f2
        axs[0, 1].plot(
            t, signal_f2,
            label=f"f2 = {f2_value} Hz",
            color="#E45756",
            linewidth=theLineWidth
        )
        axs[0, 1].set_xlabel("Time [s]")
        axs[0, 1].set_ylabel("Amplitude")
        axs[0, 1].set_title("Second Frequency Component")
        _style_axis(axs[0, 1])
        _add_legend(axs[0, 1])

        # Sum of both components
        axs[1, 0].plot(
            t, combined_signal,
            label="Signal in Time Domain",
            color="#8E5AB5",
            linewidth=theLineWidth
        )
        axs[1, 0].set_xlabel("Time [s]")
        axs[1, 0].set_ylabel("Amplitude")
        # axs[1, 0].set_title("Combined Signal in Time Domain")
        _style_axis(axs[1, 0])
        _add_legend(axs[1, 0])

        # Frequency-domain representation of the sum
        axs[1, 1].plot(
            freq[:N_current//2],
            fft_magnitude[:N_current//2],
            label="Signal in Frequency Domain",
            color="y",
            linewidth=theLineWidth
        )
        axs[1, 1].set_xlabel("Frequency [Hz]")
        axs[1, 1].set_ylabel("Magnitude")
        # axs[1, 1].set_title("Combined Signal in Frequency Domain")
        axs[1, 1].set_xlim(0, 150)
        axs[0, 0].set_ylim(-1, 1)
        axs[0, 1].set_ylim(-1, 1)
        _style_axis(axs[1, 1])
        _add_legend(axs[1, 1], loc="upper left")

        plt.tight_layout()
        plt.show()

    output = interactive_output(
        _plot,
        {"f1_value": f1_slider, "f2_value": f2_slider}
    )

    display(VBox([
        HBox([f1_slider, f2_slider]),
        output
    ]))


def show_rectangular_harmonic_widget(
    initial_pulse_width,
    pulse_generator,
    reconstruct_signal,
    reconstruct_freq_signal
):
    """Interactive rectangular-pulse Fourier approximation."""

    pulse_slider = FloatSlider(
        value=float(initial_pulse_width),
        min=0.1,
        max=0.9,
        step=0.05,
        description="Pulse width [s]",
        continuous_update=True,
        style={"description_width": "110px"},
        layout=Layout(width="350px")
    )

    harmonic_slider = IntSlider(
        value=5,
        min=1,
        max=25,
        step=1,
        description="Harmonics",
        continuous_update=True,
        style={"description_width": "80px"},
        layout=Layout(width="300px")
    )

    def _plot(pulse_width_value, num_harmonics):
        t, rect_signal, fft_result, freq = pulse_generator(pulse_width_value)

        reconstructed_signal = reconstruct_signal(fft_result, num_harmonics)
        reconstructed_freq = reconstruct_freq_signal(fft_result, num_harmonics)

        fig, (ax_time, ax_freq) = plt.subplots(1, 2, figsize=(12, 3))
        theLineWidth = 2

        ax_time.plot(
            t, rect_signal,
            label="original rectangular pulse",
            color="m",
            linewidth=theLineWidth * 2
        )
        ax_time.plot(
            t, reconstructed_signal,
            label=f"approximation with {num_harmonics} harmonics",
            color="w",
            linewidth=theLineWidth
        )
        ax_time.set_xlabel("Time [s]")
        ax_time.set_ylabel("Amplitude")
        ax_time.set_ylim([-0.2, 1.2])
        _style_axis(ax_time)
        _add_legend(ax_time, loc="center right")
        # ax_time.get_legend().get_frame().set_alpha(0.6)

        visible = (freq >= 0) & (freq <= 25)

        ax_freq.vlines(
            freq[visible],
            0,
            reconstructed_freq[visible],
            color="w",
            linewidth=1.5,
            label=f"{num_harmonics} harmonics",
        )
        ax_freq.plot(
            freq[visible],
            reconstructed_freq[visible],
            "wo",
            markersize=4,
        )

        ax_freq.set_xlabel("Frequency [Hz]")
        ax_freq.set_ylabel("Magnitude")
        ax_freq.set_ylim([0, 1.1 * np.max(reconstructed_freq[visible])])
        ax_freq.set_xlim([0, 25])

        _style_axis(ax_freq)
        _add_legend(ax_freq, loc="center right")


    output = interactive_output(
        _plot,
        {
            "pulse_width_value": pulse_slider,
            "num_harmonics": harmonic_slider
        }
    )

    display(VBox([
        HBox([pulse_slider, harmonic_slider]),
        output
    ]))


Note: you may need to restart the kernel to use updated packages.


### 1. Signals in Time and Frequency Domain

In the following, the concepts of time-domain and frequency-domain representations, and how Fourier Transforms help relate the two, are discussed. A signal in time-domain represents how the amplitude of a signal varies over time. It is the most direct way of observing a signal. For example, an audio waveform plotted as a function of time shows how the sound pressure level varies, providing the actual shape of the sound wave. A signal in frequency-domain represents the different frequency components in the signal, and their corresponding amplitudes.

Instead of illustrating how the signal changes over time, it shows how much of the signal corresponds to various frequencies (like bass, midrange, or treble in audio). A frequency-domain plot is often referred to as "spectrum".


**Declarations**

Set the parameters below and run the generation cell.

- `f1` is the first signal frequency in Hertz.  
- `f2` is the second signal frequency in Hertz.
- `pulse_width` is the width of the pulse in seconds

In [3]:
# --- Declarations: editable variables for students ---
f1 = 50
f2 = 120
pulse_width = 0.5

sampling_rate = max(f1, f2) * 10
N = sampling_rate
T = 1.0 / sampling_rate

### 2 Fourier Transform

#### 2.1 Introduction 
The Fourier Transform is a mathematical technique used to convert a time-domain signal into a frequency-domain representation. The main idea is to decompose a signal into a sum of sinusoidal components (sine and cosine waves) of different frequencies. Such a Fourier analysis is a common tool used in signal processing. It helps us analyze signals from a different perspective and manipulate them for various practical applications. 

For example, the Fourier Transform allows us to understand the frequency content of a signal, which is useful in fields like audio processing, communications, and seismology. By manipulating the frequency components (e.g., removing high frequencies for noise reduction), signals can be modified for various applications. In communication systems, a Fourier analysis helps to understand how to encode and decode information using different frequencies.

> Note: In the following subsections, we use the following notation:
- $X(f)$ represents the signal in the **frequency domain**
- $x(t)$ represents the original **time-domain** signal
- $j$ denotes the imaginary unit, with $j^2 = -1$
#### 2.2 Fourier Series 

A signal that repeats over time (a periodic signal) can be represented as an infinite sum of sine and cosine waves at different frequencies and amplitudes, which is called a Fourier Series:

$$
 x(t) = \sum_{n=-\infty}^{\infty} c_n e^{j2\pi n f_0 t},
 $$

where
- $c_n$ are complex coefficients that determine the amplitude and phase of each frequency component
- $f_0$ is the fundamental frequency of the periodic signal, with period $T_0 = 1/f_0$ 

#### 2.3 Fourier Transform (Time -> Frequency)
When dealing with non-periodic signals, the Fourier Series extends to the Fourier Transform, which can analyze signals that do not repeat. A Fourier Transform results in a continuous spectrum of the signal, showing how much of each frequency is present.


$$X(f) = \int_{-\infty}^{\infty} x(t) \cdot e^{-j2\pi ft} \, dt,$$



#### 2.4 Inverse Fourier Transform (Frequency -> Time)

The Inverse Fourier Transform converts a frequency-domain representation back to the time-domain signal $x(t)$. It simply sums up all sinusoidal components of the signal to reconstruct the original time-domain signal.

$$x(t) = \int_{-\infty}^{\infty} X(f) \cdot e^{j2\pi ft} \, df.$$



### 3. The Fast Fourier Transform


#### 3.1 Introduction
The so-called Fast Fourier Transform (FFT) computes the Discrete Fourier Transform (DFT) of a sequence  in an efficient way. It converts a time-domain signal into its constituent frequencies, providing information about the signal's frequency components. The FFT algorithm significantly reduces the computational complexity from $O(N^2)$ in the direct DFT calculation to $O(N \log N)$, making it practical for real-time applications and large datasets.


The DFT of a sequence $x[n]$ of length $N$ is given by

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j(2\pi/N)kn} \quad ,$$

where 
- $x[n]$ is the time-domain signal
- $X[k]$ is the frequency-domain representation
- $k$ is the index of the frequency component
- $j$ is the imaginary unit. 


FFT is widely used in signal processing, image analysis, audio processing, and communications for tasks such as filtering, spectral analysis, and data compression. It helps us analyze the frequency spectrum of signals, identifying periodic patterns, and isolating specific frequency components. In signal filtering, by analyzing and manipulating the frequency components, unwanted noise can be removed. During audio analysis, FFT can identify the dominant frequencies in an audio signal, aiding in tasks like pitch detection. In communication systems, FFT is used in modulation and demodulation processes.


#### 3.2 Illustration
To illustrate the use of FFT, a signal composed of two sine waves of different frequencies is used. Use the sliders below to vary the two frequencies and observe each component separately, their sum in the time domain, and the corresponding frequency-domain representation.

The slider values initially use the frequencies set in the **Declarations** section above. After running the notebook with the values required in Assignment 03, you can move the sliders to explore how changing the component frequencies affects both representations.


In [4]:
# --- Fast Fourier Transform Example ---
def generate_two_tone_signal(f1_value, f2_value):
    # Use a sampling rate that remains safely above both selected frequencies
    sampling_rate_current = max(f1_value, f2_value) * 10
    N_current = sampling_rate_current
    T_current = 1.0 / sampling_rate_current

    t = np.linspace(0.0, N_current*T_current, N_current, endpoint=False)

    signal_f1 = 0.7 * np.sin(2.0 * np.pi * f1_value * t)
    signal_f2 = np.sin(2.0 * np.pi * f2_value * t)
    combined_signal = signal_f1 + signal_f2

    fft_result = np.fft.fft(combined_signal)
    freq = np.fft.fftfreq(N_current, T_current)
    fft_magnitude = np.abs(fft_result)

    return (
        t, signal_f1, signal_f2, combined_signal,
        freq, fft_magnitude, N_current
    )


In [7]:
# --- Interactive visualization ---
show_two_frequency_fft_widget(f1, f2, generate_two_tone_signal)


#### 3.3 Approximation of a Rectangular Pulse
A rectangular pulse contains many frequency components. Keeping only a limited number of low-frequency harmonics produces an approximation of the original pulse.

Use the sliders below to vary the **number of harmonics** and the **pulse width**. The pulse-width slider initially uses the value set in the **Declarations** section. For Assignment 03, compare the approximations obtained with 5, 10, and 20 harmonics.


In [ ]:
# --- Fast Fourier Transform with Rectangular Pulse ---
t = np.linspace(0.0, N*T, N, endpoint=False)

def generate_rectangular_pulse(pulse_width_value):
    rect_signal = np.where(
        (t >= N*T*(0.5-pulse_width_value/2)) &
        (t <= N*T*(0.5+pulse_width_value/2)),
        1.0, 0.0
    )

    fft_result = np.fft.fft(rect_signal)
    freq = np.fft.fftfreq(N, T)

    return t, rect_signal, fft_result, freq


def reconstruct_signal(fft_result, num_harmonics):
    # Keep only the selected low-frequency harmonics
    filtered_fft = np.copy(fft_result)
    N_current = len(fft_result)
    filtered_fft[num_harmonics + 1:N_current - num_harmonics] = 0
    return np.fft.ifft(filtered_fft).real


def reconstruct_freq_signal(fft_result, num_harmonics):
    filtered_fft = np.copy(fft_result)
    N_current = len(fft_result)
    filtered_fft[num_harmonics + 1:N_current - num_harmonics] = 0
    return np.abs(filtered_fft)


show_rectangular_harmonic_widget(
    pulse_width,
    generate_rectangular_pulse,
    reconstruct_signal,
    reconstruct_freq_signal
)
